In [1]:
import os
import json
import ast
import numpy as np
import pandas as pd

In [ ]:
def str2vec(item) :
    # remove brackets and split by " "
    item = item[1:-1].split(" ")
    # remove " " 
    item = [eval(_item) for _item in item if _item != ""]
    
    return item
    
embedding_mapping = pd.read_csv("embedding_mapping.csv")
embedding_mapping["vector"] = embedding_mapping["vector"].apply(lambda x: str2vec(x))

In [ ]:
def str2list(item):

    try:
        return ast.literal_eval(item)
    #         return json.loads(item)
    except:
        return item
    
raw_category = pd.read_csv("output_country_flatten_archive.csv")[['CARPE_ID', 'REC_ID', 'CLIENT_ID', 'YEAR', 'MATCH_SCORE', "CLASSIFICATION.categories"]]
raw_category["CLASSIFICATION.categories"] = raw_category["CLASSIFICATION.categories"].apply(lambda x: str2list(x))

In [16]:
def category_embedding(data, embedding_key, embedding_value) :
    
    # get number of unique categories
    unique_dim = len(embedding_value[0])
    
    tmp_row = np.zeros(unique_dim)
    count = 0
    if isinstance(data, list) :
        for _item in data :
            if "category" in _item.keys() :
                try :
                    tmp_row += embedding_value[
                        embedding_key == _item["category"]
                    ].values[0]
                    count += 1
                except :
                    # non english category (สถานีวิทยุกระจายเสียง)
                    pass
    tmp_row /= count
        
    return tmp_row

In [ ]:
data_list = ["train_BG"] + ["test_BG_{}".format(i) for i in range(1, 6)]
for path in data_list:
    print(path)
    data_path = "BG_clustering/{}.csv".format(path)
    out_path = "BG_embedding/{}.csv".format(path)
    data = pd.read_csv(data_path)
    # drop previous category mapping
    data.drop(columns = ["CLASSIFICATION.categories_" + str(i) for i in range(24)], inplace = True)
    # match new embeddings
    data = data.merge(
        raw_category[['REC_ID', 'CLIENT_ID', "CLASSIFICATION.categories"]], 
        how = "left", on = ['REC_ID', 'CLIENT_ID']
    )
    data["category_embedding"] = data["CLASSIFICATION.categories"].apply(lambda x: category_embedding(
        x, embedding_mapping["unique_categories"], embedding_mapping["vector"]
    ))
    data[["CLASSIFICATION.category_embedding_" + str(i) for i in range(24)]] = data["category_embedding"].tolist()
    data.drop(columns = ["category_embedding", "CLASSIFICATION.categories"], inplace = True)
    data.to_csv(out_path, index = False)

train_BG
test_BG_1
test_BG_2
test_BG_3
test_BG_4
test_BG_5


In [ ]:
import pandas as pd
embedding_mapping = pd.read_csv("embedding_mapping.csv")